# Evaluating RAG outputs with Ragas

In [11]:
# start in root
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

### RAGAS/langchain temporary bug workaround. 
see link for more permament solution. 

https://github.com/vibrantlabsai/ragas/issues/2753#issuecomment-4563590504

In [12]:
import types
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

### Run RAG

In [13]:
from rag.pipeline import run_rag
from rag.retriever import get_retriever
from rag.ingest import *

# load PDFs
docs = load_documents()

# chunk PDFs
split_docs = split_documents(docs)

# setup system
vector_store = build_vectorstore(split_docs)
retriever = get_retriever(vector_store)

# run experiments
sample_results = [
    run_rag("What are microplastics doing to human health?", retriever),
    run_rag("How do microplastics enter the ocean?", retriever),
    run_rag("What are the effects of microplastics on the endocrine system?", retriever),
    run_rag("Have microplastics been found in human blood or tissue?", retriever),
    run_rag("What are the toxic chemicals associated with microplastic exposure?", retriever),
    run_rag("How do microplastics affect marine ecosystems?", retriever),
    run_rag("What is the role of microplastics in soil contamination?", retriever),
    run_rag("How do microplastics travel through the food chain?", retriever),
    run_rag("What are the primary sources of microplastic pollution?", retriever),
    run_rag("How do microplastics from textiles enter waterways?", retriever),
    run_rag("What is the difference between primary and secondary microplastics?", retriever),
    run_rag("What methods exist for removing microplastics from water?", retriever),
    
    # pushing scope limits - may tempt the LLM to overgeneralize
    run_rag("Do microplastics definitively cause cancer in humans?", retriever),
    run_rag("Are microplastics the leading cause of ocean pollution?", retriever),
    run_rag("Have microplastics been proven to cause infertility?", retriever),
    run_rag("Is it safe to drink tap water given microplastic contamination?", retriever),
    run_rag("How many people have died from microplastic exposure?", retriever),
    run_rag("Are microplastics more dangerous than heavy metals?", retriever),
    
    # completely out of scope - should trigger low retrieval scores
    run_rag("What is the best way to invest in renewable energy?", retriever),
    run_rag("How do I treat a jellyfish sting?", retriever),
    run_rag("What caused the 2008 financial crisis?", retriever),
    run_rag("How do vaccines work?", retriever),
]

samples = [
    {
        "id": f"q{i+1}",
        "question": result["question"],
        "contexts": result["contexts"],
        "answer": result["answer"],
    }
    for i, result in enumerate(sample_results)
]

### Inspect samples

In [14]:
from pprint import pprint
pprint(samples)

[{'answer': 'Microplastics have been associated with potential adverse effects '
            'on human health, although the research is limited. In vitro '
            'studies indicate that microplastics can cause genotoxicity and '
            'cytotoxicity, leading to increased frequencies of micronucleation '
            'and other nuclear abnormalities in human blood lymphocytes. These '
            'effects have been linked to disorders such as infertility, '
            'diabetes, obesity, and cardiovascular disease. Additionally, '
            'while there is insufficient information to draw firm conclusions '
            'about the toxicity of microplastics, some animal studies suggest '
            'that they may cause inflammation of the liver. However, these '
            'findings are based on high exposure levels that are unlikely to '
            'occur in drinking water. The health effects of biofilms that '
            'attach to microplastics in drinking water remain 

### Save samples to /data/eval

In [15]:
import json

with open("../data/eval/ragas_samples.json", "w", encoding="utf-8") as f:
    json.dump(samples, f, indent=2, ensure_ascii=False)

### Build dataset with Ragas

In [16]:
from eval.evaluation import build_ragas_dataset
dataset = build_ragas_dataset(samples)
print(dataset)

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response'], len=22)


### Run Evaluation Script
Hosted in `run_eval.py` for async scoring.
Results are stored in `/data/results/eval_results.json`

In [17]:
!python ../eval/run_eval.py

### Inspect the results

In [18]:
# read in results
import pandas as pd
df = pd.read_json("../data/results/eval_results.json")
df.head()

,question_id,faithfulness,answer_relevancy,context_relevance,scope_representation
0,q1,1.000000,0.000000,1.0,3
1,q2,1.000000,0.999995,1.0,4
2,q3,0.777778,0.945347,1.0,3
3,q4,1.000000,0.000000,1.0,4
4,q5,1.000000,0.000000,1.0,4


### Compute RAG performance from evaluation scores

In [19]:
metric_cols = ["faithfulness", "answer_relevancy", "context_relevance", "scope_representation"]

df_avg_scores = df[metric_cols].mean()

print("Evaluation Results:")
print("-------------------")
for metric, score in df_avg_scores.items():
    print(f"{metric}: {score:.4f}")

Evaluation Results:
-------------------
faithfulness: 0.9899
answer_relevancy: 0.4001
context_relevance: 0.6818
scope_representation: 3.6364
